In [2]:
import pandas as pd
df = pd.read_csv(r"C:\Users\User\OneDrive\Desktop\Project_2\ONINE_FOOD_DELIVERY_ANALYSIS.csv")

In [3]:
df.head()

,Order_ID,Customer_ID,Customer_Age,Customer_Gender,City,Area,Restaurant_ID,Restaurant_Name,Cuisine_Type,Order_Date,...,Final_Amount,Payment_Mode,Order_Status,Cancellation_Reason,Delivery_Partner_ID,Delivery_Rating,Restaurant_Rating,Order_Day,Peak_Hour,Profit_Margin
0,ORD000001,CUST6948,19.0,Male,NaN,Central,RES936,Restaurant_29,Chinese,10/20/2024,...,NaN,UPI,Delivered,NaN,DP563,5.0,4.4,Weekend,True,0.13
1,ORD000002,CUST6515,NaN,Female,Chennai,North,RES689,Restaurant_419,Chinese,8/12/2024,...,4849.0,COD,Delivered,NaN,DP369,5.0,4.7,Weekday,True,0.48
2,ORD000003,CUST1765,NaN,Male,Delhi,NaN,RES723,Restaurant_244,Arabian,12/8/2024,...,737.0,Wallet,Delivered,NaN,DP580,4.0,4.9,Weekend,True,0.08
3,ORD000004,CUST2744,NaN,Male,Mumbai,Central,RES951,Restaurant_178,Chinese,10/8/2024,...,NaN,UPI,Cancelled,Late Delivery,DP155,2.0,3.4,Weekday,NaN,0.04
4,ORD000005,CUST4389,57.0,Female,Chennai,South,RES419,Restaurant_262,Chinese,2/4/2024,...,352.0,Card,Delivered,NaN,DP728,2.0,4.4,Weekend,False,0.12


In [18]:
import pandas as pd
import numpy as np
df["Order_Date"] = pd.to_datetime(df["Order_Date"], format="%m/%d/%Y", errors="coerce")
df["Order_Weekday"] = df["Order_Date"].dt.day_name()
df["Order_Month"] = df["Order_Date"].dt.month

df = df.dropna(subset=["Order_Date"]).copy()
df = df.drop(columns=["Order_Time"], errors="ignore")
cat_cols = ["Customer_Gender", "City", "Area", "Cuisine_Type",
            "Payment_Mode", "Order_Status", "Cancellation_Reason", "Order_Day"]
for c in cat_cols:
    df[c] = df[c].str.strip().str.title()
    df["Payment_Mode"] = df["Payment_Mode"].replace({"Cod": "COD", "Upi": "UPI", "Card" : "CARD", "Wallet" :  "WALLET" })

df["Restaurant_Rating"] = df["Restaurant_Rating"].clip(upper=5)
df["Delivery_Rating"] = df["Delivery_Rating"].clip(1, 5)
df["Profit_Margin"] = df["Profit_Margin"].clip(lower=0)


df["Discount_Applied"] = df["Discount_Applied"].fillna(df["Order_Value"] - df["Final_Amount"])
df["Order_Value"] = df["Order_Value"].fillna(df["Final_Amount"].fillna(0) + df["Discount_Applied"].fillna(0))
df["Discount_Applied"] = df["Discount_Applied"].clip(upper=df["Order_Value"])
df["Final_Amount"] = df["Final_Amount"].fillna(df["Order_Value"] - df["Discount_Applied"])

df.loc[df["Final_Amount"] < 0, "Final_Amount"] = df["Order_Value"] - df["Discount_Applied"]

for c in ["Customer_Age", "Distance_km", "Order_Value", "Discount_Applied"]:
    df[c] = df[c].fillna(df[c].median())

df.loc[~cancelled, "Delivery_Time_Min"] = df.loc[~cancelled, "Delivery_Time_Min"].fillna(df["Delivery_Time_Min"].median())
df.loc[~cancelled, "Delivery_Rating"] = df.loc[~cancelled, "Delivery_Rating"].fillna(df["Delivery_Rating"].median())

for c in ["Customer_Gender", "City", "Area", "Cuisine_Type", "Payment_Mode"]:
    df[c] = df[c].fillna("Unknown")
df["Final_Amount"] = df["Final_Amount"].fillna(df["Order_Value"] - df["Discount_Applied"])

df["Cancellation_Reason"] = df["Cancellation_Reason"].fillna("Not Applicable")
df["Peak_Hour"] = (
    df["Peak_Hour"].astype("object").map({True: "Yes", False: "No"}).fillna("Unknown")
)

c = "Delivery_Time_Min"
q1, q3 = df[c].quantile([.25, .75])
iqr = q3 - q1
lo, hi = q1 - 1.5 * iqr, q3 + 1.5 * iqr

print("Fences:", lo, hi)
print("Outliers:", ((df[c] < lo) | (df[c] > hi)).sum())

df[c] = df[c].clip(lo, hi)

cancelled = df["Order_Status"] == "Cancelled"
df.loc[cancelled, ["Delivery_Time_Min", "Delivery_Rating"]] = np.nan
for c in ["Delivery_Time_Min", "Delivery_Rating"]:
    df.loc[~cancelled, c] = df.loc[~cancelled, c].fillna(df.loc[~cancelled, c].median())
delivered = df[df["Order_Status"] != "Cancelled"]

bins = [0, 45, 90, 150, 210, np.inf]
labels = ["Very Fast", "Fast", "Average", "Slow", "Very Slow"]

df["Delivery_Performance"] = pd.cut(df["Delivery_Time_Min"], bins=bins, labels=labels)

# Safely add both new categories first before applying them
df["Delivery_Performance"] = (
    df["Delivery_Performance"]
    .cat.add_categories(["Not Delivered", "Unknown"])
)

# Assign and fill values
df.loc[cancelled, "Delivery_Performance"] = "Not Delivered"
df["Delivery_Performance"] = df["Delivery_Performance"].fillna("Unknown")
df.to_csv("food_delivery_clean.csv", index=False)

Fences: -102.5 325.5
Outliers: 0


In [19]:
df

,Order_ID,Customer_ID,Customer_Age,Customer_Gender,City,Area,Restaurant_ID,Restaurant_Name,Cuisine_Type,Order_Date,...,Cancellation_Reason,Delivery_Partner_ID,Delivery_Rating,Restaurant_Rating,Order_Day,Peak_Hour,Profit_Margin,Order_Month,Order_Weekday,Delivery_Performance
0,ORD000001,CUST6948,19.0,Male,Unknown,Central,RES936,Restaurant_29,Chinese,2024-10-20,...,Not Applicable,DP563,5.0,4.4,Weekend,Unknown,0.13,10,Sunday,Slow
1,ORD000002,CUST6515,39.0,Female,Chennai,North,RES689,Restaurant_419,Chinese,2024-08-12,...,Not Applicable,DP369,5.0,4.7,Weekday,Unknown,0.48,8,Monday,Very Fast
2,ORD000003,CUST1765,39.0,Male,Delhi,Unknown,RES723,Restaurant_244,Arabian,2024-12-08,...,Not Applicable,DP580,4.0,4.9,Weekend,Unknown,0.08,12,Sunday,Slow
3,ORD000004,CUST2744,39.0,Male,Mumbai,Central,RES951,Restaurant_178,Chinese,2024-10-08,...,Late Delivery,DP155,NaN,3.4,Weekday,Unknown,0.04,10,Tuesday,Not Delivered
4,ORD000005,CUST4389,57.0,Female,Chennai,South,RES419,Restaurant_262,Chinese,2024-02-04,...,Not Applicable,DP728,2.0,4.4,Weekend,Unknown,0.12,2,Sunday,Fast
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
99994,ORD099995,CUST9174,39.0,Female,Delhi,East,RES596,Restaurant_342,Italian,2024-04-12,...,Not Specified,DP682,NaN,5.0,Weekday,Unknown,0.00,4,Friday,Not Delivered
99995,ORD099996,CUST3722,39.0,Unknown,Mumbai,Unknown,RES295,Restaurant_500,Arabian,2024-06-10,...,Not Applicable,DP835,2.0,4.0,Weekday,Unknown,0.17,6,Monday,Average
99996,ORD099997,CUST6158,39.0,Male,Mumbai,East,RES157,Restaurant_458,Arabian,2024-04-22,...,Not Applicable,DP476,4.0,3.5,Weekday,Unknown,0.00,4,Monday,Fast
99997,ORD099998,CUST2155,39.0,Unknown,Delhi,East,RES287,Restaurant_273,Unknown,2024-02-23,...,Not Specified,DP577,NaN,4.0,Weekday,Unknown,0.47,2,Friday,Not Delivered


In [20]:
df.isna().sum()

Order_ID                    0
Customer_ID                 0
Customer_Age                0
Customer_Gender             0
City                        0
Area                        0
Restaurant_ID               0
Restaurant_Name             0
Cuisine_Type                0
Order_Date                  0
Delivery_Time_Min       14886
Distance_km                 0
Order_Value                 0
Discount_Applied            0
Final_Amount                0
Payment_Mode                0
Order_Status                0
Cancellation_Reason         0
Delivery_Partner_ID         0
Delivery_Rating         14886
Restaurant_Rating           0
Order_Day                   0
Peak_Hour                   0
Profit_Margin               0
Order_Month                 0
Order_Weekday               0
Delivery_Performance        0
dtype: int64